<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Modern AI: LLMs & Generative AI** · Week 4 · Agents, Generative AI and Responsible Deployment · Lesson 22 of 24

# Open Models and Running AI Locally

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

**This lesson uses a GPU.** Before you start: **Runtime → Change runtime type → T4 GPU → Save**. The code also runs without a GPU, only more slowly.

**API key:** this lesson calls an AI model. Add your key with the **key icon (Secrets)** in the left bar, switch on **Notebook access**, and never paste a key into the code.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative).

In [ ]:
# Install the libraries this lesson needs (takes a few seconds).
%pip install -q openai

**Learning objective:** Explain open-weight models, estimate the memory a model needs, understand quantization, run an open model with Hugging Face transformers and Ollama, and choose between an API and a self-hosted model.

## Open-weight models

An **open-weight** model's trained weights can be downloaded and run on your own computer or server. Popular families include Llama, Mistral, Gemma, Qwen, DeepSeek and the small SmolLM models. Sizes range from about 100 million to hundreds of billions of parameters.

Read the **licence**: some allow any use, some restrict commercial use or very large companies, and some require attribution.

## Why run a model yourself?

| | API model | Self-hosted open model |
|---|---|---|
| Quality | Usually the best available | Good, improving fast; small models are weaker |
| Cost | Pay per token | Pay for hardware and electricity, any number of tokens |
| Privacy | Data goes to the provider (check their terms) | Data can stay on your own machine |
| Offline | No | Yes |
| Work for you | Almost none | Setup, updates, scaling, monitoring |

For a school with poor internet, sensitive records, or millions of simple requests, a small local model can be the right choice. For the hardest tasks, top API models are usually much better.

## How much memory does a model need?

Each parameter is stored as a number. Its size depends on the precision:

In [ ]:
BYTES = {"float32": 4, "float16": 2, "int8": 1, "4-bit": 0.5}

def memory_gb(params_billion, precision):
    return params_billion * 1e9 * BYTES[precision] / 1e9

print(f"{'model':>6s}" + "".join(f"{p:>10s}" for p in BYTES))
for size in [0.135, 1, 8, 70]:
    print(f"{size:>5}B" + "".join(f"{memory_gb(size, p):>9.1f}G" for p in BYTES))

An 8-billion-parameter model needs about **16 GB** in float16 but only about **4 GB** in 4-bit — the difference between needing a big GPU and fitting on a laptop. (Running also needs extra memory for the context, so add some room.)

## Quantization: smaller numbers, almost the same model

**Quantization** stores weights with fewer bits. Here is 8-bit quantization of one weight matrix:

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
W = rng.normal(0, 0.02, size=(512, 512)).astype("float32")

scale = np.abs(W).max() / 127                      # map the largest weight to 127
W_int8 = np.round(W / scale).astype("int8")        # store small integers
W_back = W_int8.astype("float32") * scale          # convert back when computing

x = rng.normal(size=512).astype("float32")
rel_error = np.linalg.norm(W @ x - W_back @ x) / np.linalg.norm(W @ x)
print(f"size: {W.nbytes / 1024:.0f} KB -> {W_int8.nbytes / 1024:.0f} KB, output error: {rel_error:.2%}")

Four times smaller, with an output error around **1%**. Real 4-bit methods (such as GPTQ, AWQ and the GGUF formats used by llama.cpp and Ollama) use cleverer scales per small group of weights to keep quality high.

## Run an open model with Hugging Face transformers

In Colab (a free GPU helps but this tiny model also runs on CPU):

In [ ]:
from transformers import AutoTokenizer, pipeline

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"        # 135 million parameters, about 270 MB
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
generator = pipeline("text-generation", model=MODEL_ID, tokenizer=tokenizer)

messages = [{"role": "user", "content": "Give one tip for saving money as a student."}]
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

out = generator(messages, max_new_tokens=60, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

The printed template shows the special tokens that mark each turn — the **chat template** you built yourself in Lesson 5. A 135-million-parameter model gives simple, sometimes wrong answers; try a 1–3 billion parameter model on a Colab GPU for much better results.

## Run models with Ollama

**Ollama** downloads quantized models and runs them on your laptop or server with one command:

```bash
# on your own computer (see ollama.com for the installer)
ollama pull llama3.2:1b
ollama run llama3.2:1b "Explain mobile money in one sentence."
```

Ollama also offers an **OpenAI-compatible API** on your machine, so the code from Week 2 works with only a different address:

In [ ]:
from openai import OpenAI

local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # the key is not checked
reply = local.chat.completions.create(model="llama3.2:1b",
                                      messages=[{"role": "user", "content": "Say hello in Kinyarwanda."}])
print(reply.choices[0].message.content)

Because the interface is the same, you can **start with an API model and later switch** some requests to a local model — or the other way round — without rewriting your app. Keep the evaluation set from Lesson 17 to check that quality is still good enough after switching.

## Small models: what to expect

- Good at: classification, extraction, short rewriting, simple chat, and tasks you have fine-tuned them for (Lesson 20).
- Weak at: long reasoning, rare facts, less common languages such as Kinyarwanda, and following many rules at once.
- Always measure on **your** test set before choosing.

## Key terms

- **Open-weight model** — a model whose weights you can download and run.
- **Precision** — how many bits store each number (32, 16, 8, 4).
- **Quantization** — storing weights with fewer bits to save memory.
- **Hugging Face transformers** — the main Python library for open models.
- **Ollama** — a tool that runs quantized models locally with an OpenAI-compatible API.
- **Self-hosting** — running the model on hardware you control.

**Takeaway:** Open models can run on your own hardware for privacy, offline use and fixed cost; quantization makes them fit, and OpenAI-compatible interfaces let you switch between local and API models while your evaluation set guards quality.

## Try it yourself

1. Use `memory_gb` to find the largest model that fits in 16 GB in 4-bit.
2. Repeat the quantization demo with 4 bits (integers from −7 to 7). How big is the error now?
3. In Colab, run SmolLM2 with three different questions. Where does it make mistakes?
4. If you have a laptop with 8 GB or more of memory, install Ollama and compare `llama3.2:1b` with an API model on five questions from your evaluation set.

---
Finished? Go back to [Modern AI: LLMs & Generative AI on WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*